# 🎯 [K14_M03] 데이터 요약 요청하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K14/k14_m03_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 병영생활·부대관리 (Level 1) / K14 군 생활 데이터 분석 |
| **미션** | 미션 03. 데이터 요약 요청하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 군 생활 데이터 분석, HITL 수기 검산, 시각화 왜곡 방지 워크북 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [Colab 환경 설정 및 헬퍼 함수] 데이터 분석, 가상 데이터 로더 & 마크다운 렌더러
# ============================================================================
!pip install -q pandas numpy tabulate matplotlib seaborn

import pandas as pd
import numpy as np
import json
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# 폰트 설정 (Colab / Linux 호환)
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# ----------------------------------------------------------------------------
# [데이터셋 자동 로드] 가상 한빛대대 4주간 생활관 생활 데이터 (12행)
# ----------------------------------------------------------------------------
data_records = [
    {'주차': '1주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 112, '싸지방': 54, 'PX': 140, '수면만족': 3.8, '식단만족': 3.9, '건의': 3},
    {'주차': '1주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 98, '싸지방': 31, 'PX': 160, '수면만족': 3.6, '식단만족': 3.5, '건의': 4},
    {'주차': '1주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 104, '싸지방': 47, 'PX': 118, '수면만족': 3.7, '식단만족': 3.2, '건의': 5},
    {'주차': '2주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 118, '싸지방': 58, 'PX': 138, '수면만족': 3.8, '식단만족': 3.8, '건의': 2},
    {'주차': '2주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 92, '싸지방': 29, 'PX': 166, '수면만족': 3.4, '식단만족': 3.4, '건의': 5},
    {'주차': '2주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 106, '싸지방': 50, 'PX': 121, '수면만족': 3.6, '식단만족': 3.1, '건의': 7},
    {'주차': '3주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 121, '싸지방': 61, 'PX': 142, '수면만족': 3.7, '식단만족': 4.0, '건의': 4},
    {'주차': '3주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 85, '싸지방': 26, 'PX': 171, '수면만족': 3.2, '식단만족': 3.4, '건의': 7},
    {'주차': '3주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 103, '싸지방': 49, 'PX': 117, '수면만족': 3.7, '식단만족': 3.3, '건의': 9},
    {'주차': '4주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 125, '싸지방': 66, 'PX': 135, '수면만족': 3.7, '식단만족': 4.1, '건의': 3},
    {'주차': '4주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 80, '싸지방': 24, 'PX': 178, '수면만족': 3.0, '식단만족': 3.3, '건의': 8},
    {'주차': '4주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 108, '싸지방': 52, 'PX': 120, '수면만족': 3.7, '식단만족': 3.2, '건의': 12}
]

df = pd.DataFrame(data_records)

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드합니다."""
    filename = f"k14_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K14", "../uploads/K14"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 분석 결과 및 검증 리포트를 시각적으로 렌더링합니다."""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 분석 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ [K14] 실습 환경, 12행 생활관 데이터셋(df) 및 시각화 렌더러 준비 완료!")
display(df)



---

## 🎯 미션 03. 데이터 요약 요청하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **AI에게 데이터 요약을 요청합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **데이터 표를 붙이고 '항목별 4주 합계·평균과 가장 높거나 낮은 중대를 표로 정리하라'는 요약 프롬프트를 설계하고 작성하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `요약 프롬프트, AI 요약 결과` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 3 데이터 요약 프롬프트 틀

| 너는 가상 한빛대대의 생활 데이터를 쉽게 설명하는 데이터 분석 도우미다.
아래 [데이터]만 사용하고, 없는 값은 만들지 마라.
항목별로 4주 합계와 주간 평균, 가장 높은 중대와 낮은 중대를 표로 정리하라.
계산 과정을 한 줄로 함께 보여라.
[데이터]
(데이터 표 붙여 넣기) |
| --- |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 프롬프트 작성(분석 요청) — 데이터를 제공하고 요약 항목과 형식을 지정해 분석을 요청했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(요약 프롬프트, AI 요약 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 03.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_03_prompt = """당신은 군 생활 데이터 분석관입니다. 아래 제공하는 한빛대대 3개 중대의 4주간 생활관 생활 데이터(12행)를 분석하여 마크다운 표로 요약해 주세요.

[요구사항]
1. 6개 항목(체력단련실, 싸지방, PX, 수면만족, 식단만족, 건의)별 4주 합계와 주간 평균을 산출해 주세요.
2. 각 항목별 4주 합계(또는 평균) 기준 최고 중대와 최저 중대를 명시해 주세요.
3. 결과는 반드시 가독성이 높은 마크다운 표 형태로 출력해 주세요."""

mission_03_result = """## 1. 표준 요약 프롬프트 전문
```text
당신은 군 생활 데이터 분석관입니다. 아래 제공하는 한빛대대 3개 중대의 4주간 생활관 생활 데이터(12행)를 분석하여 마크다운 표로 요약해 주세요.

[요구사항]
1. 6개 항목(체력단련실, 싸지방, PX, 수면만족, 식단만족, 건의)별 4주 합계와 주간 평균을 산출해 주세요.
2. 각 항목별 4주 합계(또는 평균) 기준 최고 중대와 최저 중대를 명시해 주세요.
3. 결과는 반드시 가독성이 높은 마크다운 표 형태로 출력해 주세요.
```

## 2. AI 요약 결과 기록표
| 항목 | 1중대 (합계 / 주평균) | 2중대 (합계 / 주평균) | 3중대 (합계 / 주평균) | 최고 중대 | 최저 중대 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 체력단련실(회) | 476회 / 119.0회 | 355회 / 88.8회 | 421회 / 105.3회 | 1중대 | 2중대 |
| 싸지방(회) | 239회 / 59.8회 | 110회 / 27.5회 | 198회 / 49.5회 | 1중대 | 2중대 |
| PX(회) | 555회 / 138.8회 | 675회 / 168.8회 | 476회 / 119.0회 | 2중대 | 3중대 |
| 수면만족(5점) | - / 3.75점 | - / 3.30점 | - / 3.68점 | 1중대 | 2중대 |
| 식단만족(5점) | - / 3.95점 | - / 3.40점 | - / 3.20점 | 1중대 | 3중대 |
| 건의(건) | 12건 / 3.0건 | 24건 / 6.0건 | 33건 / 8.3건 | 3중대 | 1중대 |

## 3. 도전 과제: 요약 조건 변형 실험
- **적용한 변형 조건**: (예: 1주차 대비 4주차 증감률 추가)
- **변형 요약 결과 및 평가**:"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 03. 데이터 요약 요청하기", mission_03_prompt, mission_03_result)

